# SQL Gym — 05: Time Series & Date Analysis

Practice: rolling windows, cohort analysis, gap detection, year-over-year comparisons, and anomaly detection.
Write your SQL in the `%%solution N` cell and run it — results preview inline. Then run the check cell to validate.

**Tables:** `users`, `accounts`, `merchants`, `transactions`

In [ ]:
from pathlib import Path
import sys

_cwd = Path.cwd()
_candidates = [_cwd / "sql", _cwd, _cwd.parent, _cwd.parent / "sql",
               _cwd.parent.parent, _cwd.parent.parent / "sql"]
_sql_dir = next((p for p in _candidates if (p / "utils" / "__init__.py").exists()), None)
if _sql_dir is None:
    raise RuntimeError(
        "Cannot locate sql/utils. Run: uv run jupyter lab from the project root."
    )
if str(_sql_dir) not in sys.path:
    sys.path.insert(0, str(_sql_dir))

DATA_DIR = _sql_dir / "data"

from utils import get_conn, check, register_sql_magic
from utils.checks.time_series import Checker

conn = get_conn(DATA_DIR)
checker = Checker(conn)
register_sql_magic()
print("Ready. Tables: users, merchants, accounts, transactions")

In [ ]:
for table in ["users", "merchants", "accounts", "transactions"]:
    print(f"\n{'─'*50}\n  {table}\n{'─'*50}")
    display(conn.execute(f"SELECT * FROM {table} LIMIT 3").df())
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  ({n:,} rows total)")

## Problem 1: Daily Volume with 7-Day Rolling Average

Compute daily completed transaction count and total amount, along with a 7-day rolling average of the daily amount (including the current day).

<details>
<summary>Hint</summary>

Aggregate to daily totals in a CTE first. Then apply `AVG(daily_amount) OVER (ORDER BY txn_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`. The `ROWS BETWEEN` spec is standard across DuckDB, Snowflake, BigQuery, and PostgreSQL. `RANGE BETWEEN` uses value-based framing — use `ROWS` for row-count-based windows.

</details>

| Column | Type | Notes |
|--------|------|-------|
| txn_date | date | |
| daily_count | bigint | |
| daily_amount | double | 2 decimal places |
| rolling_7d_avg | double | 2 decimal places, 7-day rolling average |

Expected: one row per day that has transactions, ordered by txn_date ASC.

In [ ]:
%%solution 1

In [ ]:
checker.p1(solution_1)  # type: ignore[name-defined]  # noqa: F821

## Problem 2: Monthly First-Transaction Cohorts

For each calendar month, count how many users made their very first completed transaction during that month. These are monthly acquisition cohorts.

<details>
<summary>Hint</summary>

CTE to find `MIN(txn_date)` per user (joining transactions to accounts to get user_id). Then group by `DATE_TRUNC('month', first_txn_date)::DATE`. This is the basis for cohort retention analysis — the cohort definition step.

</details>

| Column | Type | Notes |
|--------|------|-------|
| cohort_month | date | month of user's first transaction |
| new_active_users | bigint | users making their first transaction in this month |

Expected: one row per month that had at least one new active user, ordered by cohort_month ASC.

In [ ]:
%%solution 2

In [ ]:
checker.p2(solution_2)  # type: ignore[name-defined]  # noqa: F821

## Problem 3: Accounts with Long Inactivity Gaps

Find all instances where an account had a gap of more than 30 days between consecutive completed transactions. Return the account, the gap start date, end date, and the gap length in days.

<details>
<summary>Hint</summary>

Use `LAG(txn_date) OVER (PARTITION BY account_id ORDER BY txn_date, txn_id)` to get the previous transaction's date. Compute `txn_date - prev_txn_date` for the gap. Filter `WHERE gap_days > 30`. In DuckDB and PostgreSQL, date subtraction returns an integer (days). Sort by gap_days DESC to surface the longest gaps first.

</details>

| Column | Type | Notes |
|--------|------|-------|
| account_id | integer | |
| gap_start | date | date of the transaction before the gap |
| gap_end | date | date of the transaction after the gap |
| gap_days | integer | length of the gap in days |

Expected: variable rows (all gaps > 30 days), ordered by gap_days DESC, account_id ASC.

In [ ]:
%%solution 3

In [ ]:
checker.p3(solution_3)  # type: ignore[name-defined]  # noqa: F821

## Problem 4: Year-over-Year Revenue by MCC Category

For each MCC category and year, compute total completed debit revenue. Add columns for the previous year's revenue and the year-over-year percentage change.

<details>
<summary>Hint</summary>

CTE for annual totals per category. Outer query adds `LAG(total_revenue) OVER (PARTITION BY mcc_category ORDER BY year)`. YoY % = `(current - prev) / prev * 100`. The first year per category will have NULL for the lag and pct change columns. `EXTRACT(YEAR FROM txn_date)::INT` gives integer year in DuckDB and PostgreSQL. Snowflake: `YEAR(txn_date)`. BigQuery: `EXTRACT(YEAR FROM txn_date)`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| mcc_category | string | |
| year | integer | calendar year |
| total_revenue | double | 2 decimal places |
| prev_year_revenue | double | 2 decimal places, NULL for first year |
| yoy_pct_change | double | %, 2 decimal places, NULL for first year |

Expected: rows ordered by mcc_category ASC, year ASC.

In [ ]:
%%solution 4

In [ ]:
checker.p4(solution_4)  # type: ignore[name-defined]  # noqa: F821

## Problem 5: Transaction Volume Anomalies

Identify days where the completed transaction count was more than 2 standard deviations above the daily mean. Return the date, count, total amount, and the global mean and stddev for context.

<details>
<summary>Hint</summary>

CTE for daily stats. CROSS JOIN with a single-row stats CTE (`AVG`, `STDDEV`). Filter `WHERE daily_count > mean_daily_count + 2 * std_daily_count`. `STDDEV()` (population stddev) works the same in DuckDB, Snowflake, BigQuery, and PostgreSQL. `CROSS JOIN` of a scalar CTE is idiomatic for broadcasting constants.

</details>

| Column | Type | Notes |
|--------|------|-------|
| txn_date | date | |
| daily_count | bigint | |
| daily_amount | double | 2 decimal places |
| mean_daily_count | double | 2 decimal places, global mean |
| std_daily_count | double | 2 decimal places, global standard deviation |

Expected: variable rows (anomaly days only), ordered by daily_count DESC.

In [ ]:
%%solution 5

In [ ]:
checker.p5(solution_5)  # type: ignore[name-defined]  # noqa: F821